# YOLOv5 历史抽帧筛选与微调说明

本次离线处理基于现有 VOC 风格数据集：`F:/1/labelimg/data/yolo_data`。

实际执行结果：

- 原始 `images/annotations`：`67462`
- 按 `keep_every=3` 保留后：`22489`
- 新训练集目录：`F:/1/labelimg/data/yolo_data_stride3`
- `train.txt`：`18438` 张
- `val.txt`：`4051` 张
- 验证集按视频级划分，当前保留 `video13_mp4` 作为验证集，其余视频进入训练集


## 1. 已执行的离线筛选命令

```powershell
D:\Miniconda3\python.exe F:\1\yolov5-master\tools\label_tools.py build-trainable-voc \
  --dataset-root F:\1\labelimg\data\yolo_data \
  --output-root F:\1\labelimg\data\yolo_data_stride3 \
  --data-yaml F:\1\yolov5-master\data\dataAirVis.yaml \
  --keep-every 3 \
  --offset 0 \
  --val-keys video13_mp4
```


### 离线筛选命令参数说明

- `build-trainable-voc`：一条命令完成三件事，先筛帧，再从 XML 重建 YOLO 标签，最后写 `train.txt` / `val.txt` / `data.yaml`
- `--dataset-root`：原始 VOC 数据根目录，要求至少包含 `images/` 和 `annotations/`
- `--output-root`：筛选后的新数据集目录；本次没有覆盖原始数据，而是写到新目录
- `--data-yaml`：类别定义来源；这里复用仓库的 `dataAirVis.yaml`，确保 15 类映射一致
- `--keep-every 3`：每 3 帧保留 1 帧
- `--offset 0`：每组三帧保留第 1 张；如果想保留第 2 张，可改成 `--offset 1`
- `--val-keys video13_mp4`：按视频级划分验证集，避免同一视频的近邻帧同时落入 train 和 val


## 2. 基于筛选后训练集做 `checkpoint/yolov5_best.pt` 微调

### 2.1 训练前冒烟检查

```powershell
D:\Miniconda3\python.exe F:\1\yolov5-master\train.py \
  --data F:\1\labelimg\data\yolo_data_stride3\data.yaml \
  --weights F:\1\yolov5-master\checkpoint\yolov5_best.pt \
  --epochs 1 \
  --batch-size 4 \
  --imgsz 640 \
  --device 0 \
  --workers 4 \
  --seed 0 \
  --project F:\1\yolov5-master\runs\train \
  --name stride3_sanity_check
```

### 2.2 正式微调命令

```powershell
D:\Miniconda3\python.exe F:\1\yolov5-master\train.py \
  --data F:\1\labelimg\data\yolo_data_stride3\data.yaml \
  --weights F:\1\yolov5-master\checkpoint\yolov5_best.pt \
  --epochs 30 \
  --batch-size 8 \
  --imgsz 640 \
  --device 0 \
  --workers 4 \
  --seed 0 \
  --project F:\1\yolov5-master\runs\train \
  --name ft_yolo_data_stride3_bs8
```


### 微调命令参数说明

- `--data`：筛选后数据集的配置文件；当前文件已经指向 `train.txt` / `val.txt`
- `--weights`：微调起点；这里用现有 `checkpoint/yolov5_best.pt`，不是从头训练
- `--epochs`：训练轮数；建议先用 `1` 做链路体检，再用 `30` 做正式微调
- `--batch-size`：单次训练批大小；当前机器是 GTX 1660 6GB，先保守用 `8`，若 OOM 降到 `4`
- `--imgsz`：输入尺寸；继续用当前分支常用的 `640`
- `--device 0`：使用第 0 块 GPU
- `--workers 4`：DataLoader 线程数；对当前 Windows 机器先用较保守的值
- `--seed 0`：固定随机种子，方便复现
- `--project` / `--name`：训练输出目录，结果会落到 `runs/train/<name>/`


## 3. 后续新视频的推荐推理命令

后续不建议再先全量抽帧、再事后删帧。新视频建议直接从源头降采样：

```powershell
D:\Miniconda3\python.exe F:\1\yolov5-master\detect.py \
  --weights F:\1\yolov5-master\checkpoint\yolov5_best.pt \
  --source F:\1\video\output \
  --data F:\1\yolov5-master\data\dataAirVis.yaml \
  --imgsz 640 \
  --device 0 \
  --save-img-frames \
  --voc-root F:\1\labelimg\data\yolo_data_stride3_next \
  --vid-stride 3 \
  --project F:\1\yolov5-master\runs\detect \
  --name stride3_detect
```

关键点：`--vid-stride 3` 是从读取视频阶段就每 3 帧取 1 帧，比“先全跑再删图”更省推理时间。
